# Agents

- Tools become truly powerful when combined with agents. 
- An agent is an LLM that reasons about WHICH tool to call, calls it, observes the result, and repeats until it can answer the user's question.

## Build a simple agent

In [37]:
!pip install -U langchain langchain-core langchain-openai langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.2 MB/s  0:00:00m eta 0:00:01
  Attempting uninstall: langchain-classic
    Found existing installation: langchain-classic 1.0.1
    Uninstalling langchain-classic-1.0.1:
      Successfully uninstalled langchain-classic-1.0.1


In [1]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv
import os
load_dotenv()

# Step 1: Define tools
@tool
def add(a: int, b: int) -> int:
    """Add two integers together. Use for addition tasks."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers. Use for multiplication tasks."""
    return a * b

@tool
def get_country_capital(country: str) -> str:
    """Returns the capital city of a given country."""
    capitals = {
        'france': 'Paris', 'japan': 'Tokyo',
        'india': 'New Delhi', 'uae': 'Abu Dhabi',
    }
    return capitals.get(country.lower(), 'Unknown')

tools = [add, multiply, get_country_capital]

# Step 2: LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, 
                 api_key=os.getenv('OPENAI_API_KEY'), base_url=os.getenv('OPENAI_BASE_URL'))

# Step 3: Create agent (no prompt template needed)
agent = create_react_agent(
    llm,
    tools,
    prompt=SystemMessage("You are a helpful assistant. Answer questions using the available tools.")
)

# Step 4: Run 
result = agent.invoke({
    "messages": [("human", "What is the capital of France? Also, what is 15 multiplied by 7?")]
})

print(result["messages"][-1].content)

/Users/gourasundarmohanty/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/bj/3vg2xkms7sx_964f4zt1gg1r0000gn/T/ipykernel_26957/927136969.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


The capital of France is Paris, and 15 multiplied by 7 is 105.


## Build an agent using a prompt template + 2 tools (e.g., calculator + search)

In [2]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_core.runnables import RunnableLambda, RunnableBranch

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

import os 
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [3]:
# ------------------------------------------------------------
# Step 1: LLM
# ------------------------------------------------------------
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, 
                 api_key=os.getenv('OPENAI_API_KEY'), base_url=os.getenv('OPENAI_BASE_URL'))

In [4]:
# ------------------------------------------------------------
# Step 2: Tools
# ------------------------------------------------------------
calculator = Tool(
    name="Calculator",
    func=lambda x: str(eval(x)),
    description="Compute math expressions such as '12 * 7 + 5'"
)

wiki_runner = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wikipedia = Tool(
    name="Wikipedia",
    func=wiki_runner.run,
    description="Search Wikipedia for factual information."
)

In [5]:
# ------------------------------------------------------------
# Step 3: Prompt for tool decision
# ------------------------------------------------------------
decision_prompt = PromptTemplate.from_template("""
You are a tool-using agent.

TOOLS:
- Calculator → for math
- Wikipedia → for factual answers

Question: {question}

Decide action using EXACTLY one of:
1. TOOL=Calculator | INPUT=...
2. TOOL=Wikipedia | INPUT=...
3. NO_TOOL

Respond with ONLY one line.
""")

In [10]:
# Runnable: ask LLM what to do
decide_tool = decision_prompt | llm | RunnableLambda(lambda msg: msg.content)

In [13]:
# Why RunnableLambda - StringParser
result = (decision_prompt | llm).invoke({"question": "What is 5 * 6?"})
print(type(result))   # <class 'langchain_core.messages.ai.AIMessage'>
print("Result without RunnableLambda", result)   

# Why RunnableLambda - StringParser
result_1 = (decision_prompt | llm| RunnableLambda(lambda msg: msg.content)).invoke({"question": "What is 5 * 6?"})
print(type(result_1))   # String
print("Result with RunnableLambda", result_1)     

<class 'langchain_core.messages.ai.AIMessage'>
Result without RunnableLambda content='TOOL=Calculator | INPUT=5 * 6' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 79, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7e09045385', 'id': 'chatcmpl-DGzcF7AaMkz8jLLP0Z5C5wJnk3CJo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ccb9b-2ed0-7ec3-b100-70afe98fd784-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 79, 'output_tokens': 11, 'total_tokens': 90, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
<class 'str'>
Result with RunnableLambda T

In [7]:
# ------------------------------------------------------------
# Step 4: Tool execution handler
# ------------------------------------------------------------
def run_tool(data):
    decision = data["decision"]

    if decision.startswith("TOOL=Calculator"):
        expr = decision.split("INPUT=")[-1].strip()
        return calculator.run(expr)

    if decision.startswith("TOOL=Wikipedia"):
        query = decision.split("INPUT=")[-1].strip()
        return wikipedia.run(query)

    return None


tool_executor = RunnableLambda(run_tool)

In [8]:
# ------------------------------------------------------------
# Step 5: Fallback LLM (no tool needed)
# ------------------------------------------------------------
direct_answer = PromptTemplate.from_template("""
Answer the following question directly:

{question}
""") | llm | RunnableLambda(lambda msg: msg.content)

In [11]:
# ------------------------------------------------------------
# Step 6: Build the final agent
# ------------------------------------------------------------
agent = (
    # Pass question forward
    {"question": lambda x: x["question"], 
     "decision": decide_tool}
    |
    RunnableBranch(
        (lambda x: x["decision"].startswith("TOOL="), tool_executor),
        direct_answer
    )
)

In [40]:
# ------------------------------------------------------------
# Step 7: TEST
# ------------------------------------------------------------
print("\n--- Wikipedia Query ---")
print(agent.invoke({"question": "Who is the president of France?"}))

print("\n--- Math Query ---")
print(agent.invoke({"question": "12 * 7 + 5"}))

print("\n--- Direct Answer Query ---")
print(agent.invoke({"question": "Explain machine learning in one sentence."}))


--- Wikipedia Query ---
Page: President of France
Summary: The president of France, officially the president of the French Republic (French: Président de la République française, [pʁezidɑ̃ d(ə) la ʁepyblik fʁɑ̃sɛːz]) or president of the Republic (Président de la République), is the executive head of state of France, and the commander-in-chief of the French Armed Forces. As the presidency is the supreme magistracy of the country, the position is the highest office in France. The powers, functions and duties of prior presidential offices, in addition to their relation with the prime minister and government of France, have over time differed with the various constitutional documents since the Second Republic.
The president of France is the ex officio co-prince of Andorra, grand master of the Legion of Honour and of the National Order of Merit, and protector of the Institut de France in Paris. The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome,